# Day 041 — Exercise 4: run_pandas_code

**What you'll build:** `run_pandas_code(code, df) -> str` — safely exec a string of Python code in a namespace that has `df` and `pd` available, read the `result` variable, and return it as a string.

**Why it matters:** `exec()` in a controlled namespace is how you run LLM-generated code without exposing your full Python environment. The namespace dict acts as a sandbox: the code can only see `df` and `pd` (plus Python builtins). Wrapping exec in try/except means a bad code generation returns an informative error string instead of crashing the pipeline.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import re
import ollama
import pandas as pd
import io


RETAIL_CSV = (
    'order_id,product,category,region,price,quantity\n'
    '1,Widget,Electronics,North,25.0,10\n'
    '2,Gadget,Electronics,South,150.0,3\n'
    '3,Widget,Electronics,South,25.0,5\n'
    '4,Doohickey,Accessories,East,8.0,50\n'
    '5,Gadget,Electronics,East,150.0,7\n'
    '6,Widget,Electronics,East,25.0,4\n'
    '7,Doohickey,Accessories,North,8.0,20\n'
    '8,Gadget,Electronics,North,150.0,2\n'
    '9,Widget,Electronics,West,25.0,6\n'
    '10,Doohickey,Accessories,South,8.0,15\n'
    '11,Thingamajig,Accessories,North,200.0,1\n'
    '12,Thingamajig,Accessories,East,200.0,4'
)
SALES_DF = pd.read_csv(io.StringIO(RETAIL_CSV))
SALES_DF['revenue'] = SALES_DF['price'] * SALES_DF['quantity']


def get_df_schema(df) -> str:
    lines = [f"Shape: {df.shape[0]} rows x {df.shape[1]} columns"]
    lines.append("\nColumns and dtypes:")
    for col, dtype in df.dtypes.items():
        lines.append(f"  {col}: {dtype}")
    lines.append(f"\nSample (first 3 rows):\n{df.head(3).to_string(index=False)}")
    return "\n".join(lines)


def build_query_prompt(question: str, schema_str: str) -> str:
    return (
        "You are a Python data analyst. Write pandas code to answer the question.\n\n"
        "Requirements:\n"
        "- The DataFrame is already loaded as `df`. `pd` is also in scope.\n"
        "- Store the final answer in a variable named `result`.\n"
        "- Respond with ONLY a fenced Python code block, no explanation.\n\n"
        f"DataFrame schema:\n{schema_str}\n\n"
        f"Question: {question}"
    )


import re

def extract_code(response: str) -> str:
    fence = '`' * 3
    match = re.search(fence + r'python\s*(.*?)' + fence, response, re.DOTALL)
    if match:
        return match.group(1).strip()
    match = re.search(fence + r'\s*(.*?)' + fence, response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()

## Your Implementation

In [ ]:
import pandas as pd

def run_pandas_code(code: str, df) -> str:
    """
    Execute a pandas code string and return the result as a string.

    - Create namespace = {'df': df, 'pd': pd}
    - exec(code, namespace) wrapped in try/except Exception
    - On exception: return f'Code execution error: {e}'
    - After exec: result = namespace.get('result', 'No result variable found')
    - Return str(result)

    Returns:
        str — str(result) from the exec'd code, or error message
    """
    # TODO: build namespace dict with df and pd
    # TODO: try exec(code, namespace), except Exception as e: return error str
    # TODO: get result from namespace, return str(result)
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: function defined
    try:
        assert 'run_pandas_code' in globals()
        passed += 1; print('\u2705 Check 1: run_pandas_code is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns a string
    try:
        _r = run_pandas_code('result = df.shape[0]', SALES_DF)
        assert isinstance(_r, str), \
            f'expected str, got {type(_r).__name__}'
        passed += 1; print('\u2705 Check 2: returns a string')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: correctly evaluates row count
    try:
        assert _r == '12', \
            f'df.shape[0] should be 12, got {repr(_r)}'
        passed += 1; print('\u2705 Check 3: df.shape[0] → "12" correct')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: executes pandas aggregation correctly
    try:
        _rev = run_pandas_code("result = df['revenue'].sum()", SALES_DF)
        assert '4105' in _rev, \
            f'revenue sum should contain 4105, got {repr(_rev)}'
        passed += 1; print(f'\u2705 Check 4: revenue sum correct ({_rev})')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: returns error message on bad code (not an exception)
    try:
        _err = run_pandas_code('result = df["nonexistent_col"].sum()', SALES_DF)
        assert 'error' in _err.lower() or 'Error' in _err, \
            f'bad code should return error string, got {repr(_err)}'
        passed += 1; print('\u2705 Check 5: bad code returns error string')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import pandas as pd

def run_pandas_code(code: str, df) -> str:
    namespace = {'df': df, 'pd': pd}
    try:
        exec(code, namespace)
    except Exception as e:
        return f"Code execution error: {e}"
    result = namespace.get('result', 'No result variable found')
    return str(result)
```

</details>